# Tiền xử lý dữ liệu — Thành viên 3

Input: `raw_news.csv` (hoặc `news_data.json`) từ Thành viên 1.
Output: `clean_news.csv` — dữ liệu đã làm sạch, có thêm cột `tokens` (đã tách từ, bỏ stopwords) để Thành viên 4 dùng đếm tần suất từ khóa.

Các bước: (1) xóa thiếu/trùng → (2) xóa HTML/ký tự đặc biệt → (3) tách từ tiếng Việt → (4) loại bỏ stopwords → (5) xuất file.


In [ ]:
# Cài thư viện cần thiết (chạy 1 lần)
# !pip install pandas underthesea


In [ ]:
import pandas as pd
import re

# --- Đọc dữ liệu ---
# Khi đã có file thật từ Thành viên 1, dùng dòng này:
df = pd.read_csv("raw_news.csv")

# Demo tạm với dữ liệu mẫu (xóa đoạn này khi đã có raw_news.csv thật)
# df = pd.DataFrame([
#     {"title": "<b>AI</b> bùng nổ tại Việt Nam", "publish_date": "2026-01-05",
#      "author": "Nguyễn Văn A", "content": "<p>Trí tuệ nhân tạo đang phát triển rất nhanh &amp; mạnh.</p>",
#      "comments": 12, "source": "VnExpress", "url": "http://a.com/1"},
# ])

print("Số dòng ban đầu:", len(df))
df.head()


## 1. Xóa dữ liệu thiếu và trùng lặp

In [ ]:
# Xóa các dòng thiếu title hoặc content
df = df.dropna(subset=["title", "content"])
df = df[df["content"].astype(str).str.strip() != ""]

# Xóa trùng lặp theo url (ưu tiên), nếu không có url thì theo title
if "url" in df.columns:
    df = df.drop_duplicates(subset="url")
else:
    df = df.drop_duplicates(subset="title")

df = df.reset_index(drop=True)
print("Số dòng sau khi xóa thiếu/trùng:", len(df))


## 2. Xóa HTML và ký tự đặc biệt

In [ ]:
def clean_text(text):
    text = str(text)
    text = re.sub(r"<[^>]+>", " ", text)      # xóa tag HTML
    text = re.sub(r"&\w+;", " ", text)        # xóa HTML entity (&amp; &nbsp;...)
    text = re.sub(r"http\S+|www\.\S+", " ", text)  # xóa link
    text = re.sub(r"[^\w\sÀ-ỹ]", " ", text)   # xóa ký tự đặc biệt, GIỮ dấu tiếng Việt
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["title_clean"] = df["title"].apply(clean_text)
df["content_clean"] = df["content"].apply(clean_text)
df[["title_clean", "content_clean"]].head()


## 3. Tách từ tiếng Việt

Dùng `underthesea` (có thể đổi sang `pyvi` nếu muốn, API tương tự).

In [ ]:
from underthesea import word_tokenize

def tokenize(text):
    return word_tokenize(text.lower(), format="text").split()

df["tokens"] = df["content_clean"].apply(tokenize)
df["tokens"].head()


## 4. Loại bỏ stopwords

Dùng file `stopwords_vi.txt` (mỗi dòng 1 từ/cụm từ).

In [ ]:
with open("stopwords_vi.txt", encoding="utf-8") as f:
    stopwords = set(line.strip() for line in f if line.strip())

def remove_stopwords(tokens):
    return [t for t in tokens if t.replace("_", " ") not in stopwords]

df["tokens"] = df["tokens"].apply(remove_stopwords)
df["tokens"].head()


## 5. Xuất dữ liệu đã làm sạch

In [ ]:
# Gộp tokens thành chuỗi để lưu vào CSV (đọc lại bằng .split())
df["tokens_str"] = df["tokens"].apply(lambda toks: " ".join(toks))

output_cols = ["title", "title_clean", "publish_date", "author",
               "content_clean", "tokens_str", "comments", "source", "url"]
output_cols = [c for c in output_cols if c in df.columns]

df[output_cols].to_csv("clean_news.csv", index=False, encoding="utf-8-sig")
print("Đã lưu clean_news.csv —", len(df), "dòng")


## Việc cần trao đổi với nhóm

- **Thành viên 2**: import script hiện đang đọc trực tiếp `news.csv` / `news_data.json` để insert vào MySQL, chưa qua bước làm sạch. Đề xuất đổi sang đọc `clean_news.csv` (file này) trước khi import.
- **Thành viên 4**: cột `tokens_str` ở đây đã tách từ + bỏ stopwords sẵn, dùng để đếm tần suất từ khóa (`Counter`) mà không cần tách lại.
- **Thành viên 1**: xác nhận tên file output thực tế là gì (`raw_news.csv`?) để khớp với dòng đọc file ở Cell 2.
